#### 1. CONEXÃO COM POSTGRESQL

In [1]:
from dotenv import load_dotenv
from sqlalchemy import create_engine
import os

load_dotenv()

DATABASE_URL = (
    f"postgresql+psycopg://"
    f"{os.getenv('DB_USER')}:"
    f"{os.getenv('DB_PASSWORD')}@"
    f"{os.getenv('DB_HOST')}:"
    f"{os.getenv('DB_PORT')}/"
    f"{os.getenv('DB_NAME')}"
)

engine = create_engine(DATABASE_URL)

print("Conexão criada.")

Conexão criada.


#### 2. TESTE LEITURA DE VARIÁVEIS

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

print("Usuário:", os.getenv("DB_USER"))
print("Host:", os.getenv("DB_HOST"))
print("Banco:", os.getenv("DB_NAME"))

Usuário: postgres
Host: localhost
Banco: comex_dw


#### 3. CONEXÃO COM O BANCO

In [17]:
from dotenv import load_dotenv
from sqlalchemy.engine import URL
from sqlalchemy import create_engine, text
import os

load_dotenv(override=True)

connection_url = URL.create(
    drivername="postgresql+psycopg",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME")
)

engine = create_engine(connection_url)

with engine.connect() as conn:
    resultado = conn.execute(
        text("SELECT current_database(), current_user")
    )

    print(resultado.fetchone())

('comex_dw', 'postgres')


#### 4. CARREGAR TABELAS DIMENSÃO 

In [ ]:
import pandas as pd 

df_estado = pd.read_parquet("../data/processed/dim_estado.parquet")
df_ncm = pd.read_parquet("../data/processed/dim_ncm.parquet")
df_pais = pd.read_parquet("../data/processed/dim_pais.parquet")
df_tempo = pd.read_parquet("../data/processed/dim_tempo.parquet")

df_pais.to_sql(
    "dim_pais",
    con=engine,
    if_exists="append",
    index=False
)

df_ncm.to_sql(
    "dim_ncm",
    con=engine,
    if_exists="append",
    index=False
)

df_estado.to_sql(
    "dim_estado",
    con=engine,
    if_exists="append",
    index=False
)

df_tempo.to_sql(
    "dim_tempo",
    con=engine,
    if_exists="append",
    index=False
)


-1

#### 5. CARREGAR TABELA FATO 

In [25]:
df_fato = pd.read_parquet("../data/processed/fact_exportacao.parquet")

df_fato.to_sql(
    "fact_exportacao",
    con=engine,
    if_exists="append",
    index=False
)

-1

#### 6. VALIDAÇÃO 

In [27]:
from sqlalchemy import text

with engine.connect() as conn:
    for tabela in ["dim_estado", "dim_ncm", "dim_pais", "dim_tempo", "fact_exportacao"]:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {tabela}"))
        print(tabela, result.fetchone()[0])

dim_estado 34
dim_ncm 8811
dim_pais 747
dim_tempo 41
fact_exportacao 5561220


#### 7. CRIAÇÃO DE VIEWS

In [29]:
with engine.connect() as conn:
    conn.execute(text("""
        CREATE OR REPLACE VIEW vw_exportacao_detalhada AS
        SELECT
            t.co_ano,
            t.co_mes,
            t.trimestre,

            p.nome_pais,

            e.estado,
            e.regiao,
            e.sg_uf_ncm,

            n.co_ncm,
            n.descricao_produto,

            f.vl_fob,
            f.kg_liquido,
            f.valor_medio_kg

        FROM fact_exportacao f
        JOIN dim_tempo t  ON f.id_tempo = t.id_tempo
        JOIN dim_pais p   ON f.id_pais = p.id_pais
        JOIN dim_estado e ON f.id_estado = e.id_estado
        JOIN dim_ncm n    ON f.id_ncm = n.id_ncm;
    """))
    conn.commit()

In [30]:
with engine.connect() as conn:
    conn.execute(text("""
        CREATE OR REPLACE VIEW vw_exportacao_por_pais AS
        SELECT
            p.nome_pais,

            COUNT(*) AS total_registros,
            SUM(f.vl_fob) AS valor_total_exportado,
            SUM(f.kg_liquido) AS peso_total,

            AVG(f.valor_medio_kg) AS preco_medio_kg

        FROM fact_exportacao f
        JOIN dim_pais p ON f.id_pais = p.id_pais

        GROUP BY p.nome_pais;
    """))
    conn.commit()